In [ ]:
!pip install -q datasets==2.18.0 tqdm

In [ ]:
!pip install torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121  # or cu118 / cpu
!pip install datasets>=2.18 transformers>=4.40

In [ ]:
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from tqdm.auto import tqdm
import random, os, shutil

SEED          = 42          # change for a different shuffle
TEST_FRAC     = 0.3         # 10 % of threads → test
DATASET_NAME  = "rickRossie/bluemoon_roleplay_chat_data_300k_messages"

random.seed(SEED)

In [ ]:
from datetime import datetime
from random import randint

# ------------------------------------------------------------------
# 1 · Stream & bucket messages per thread

In [ ]:
raw = load_dataset(DATASET_NAME, split="train", streaming=True)

threads = {}
for row in tqdm(raw, desc="Grouping"):
    txt = row["message"]
    if not txt:
        continue
    threads.setdefault(row["thread_title"], []).append(
        (row["message_timestamp"], txt)
    )


In [ ]:
# 1. Basic size stats
print("Total threads:", len(threads))
lengths = [len(msgs) for msgs in threads.values()]
print(f"Min thread length: {min(lengths)}")
print(f"Max thread length: {max(lengths)}")
print(f"Avg thread length: {sum(lengths)/len(lengths):.2f}")

# 2. Sample a thread and inspect
sample_title = random.choice(list(threads.keys()))
sample_msgs = sorted(threads[sample_title], key=lambda x: x[0])

print(f"\n▶ Thread title: {sample_title}")
print(f"Number of messages: {len(sample_msgs)}\n")
for ts, msg in sample_msgs[:5]:
    print(msg[:100].replace("\n", " "), "…")

In [ ]:
threads = {k: v for k, v in threads.items() if len(v) >= 4}

print("Threads after filtering:", len(threads))

In [ ]:
lengths = [len(msgs) for msgs in threads.values()]
print(f"Min thread length: {min(lengths)}")
print(f"Max thread length: {max(lengths)}")
print(f"Avg thread length: {sum(lengths)/len(lengths):.2f}")


# ------------------------------------------------------------------
# 2 · Train / test split at the thread level

In [ ]:
titles = list(threads.keys())
random.shuffle(titles)

cut = int(len(titles) * (1 - TEST_FRAC))
train_titles, test_titles = set(titles[:cut]), set(titles[cut:])

def make_pairs(title_set):
    for tt in title_set:
        turns = sorted(threads[tt], key=lambda x: x[0])
        msgs  = [m for _, m in turns]
        for i in range(len(msgs) - 3):
            yield {
                "input_text":  "\n\n".join(msgs[i:i+3]),
                "target_text": msgs[i+3],
            }

train_pairs = Dataset.from_generator(lambda: make_pairs(train_titles))
test_pairs  = Dataset.from_generator(lambda: make_pairs(test_titles))

In [ ]:
assert train_titles.isdisjoint(test_titles), "Thread titles leak between train and test"
print("✅ Train/test split is disjoint.")


In [ ]:
# Pick a thread from train
some_train_title = random.choice(list(train_titles))
thread = threads[some_train_title]

# Show timestamps before sorting
print(f"\n▶ Raw order of thread: {some_train_title}")
for ts, msg in thread[:5]:
    print(ts, "→", msg[:60].replace("\n", " "))

# Date format used in your thread: "Aug 10, 2016 at 5:28 AM"
fmt = "%b %d, %Y at %I:%M %p"

# Sort using parsed datetime
sorted_turns = sorted(thread, key=lambda x: datetime.strptime(x[0], fmt))

print(f"\n▶ After sorting:")
for ts, msg in sorted_turns[:5]:
    dt = datetime.strptime(ts, fmt)
    print(dt, "→", msg[:60].replace('\n',' '))


In [ ]:
# Sample a thread title
title = random.choice(list(train_titles))
turns = threads[title]

# Sort and extract messages only
msgs = [m for _, m in sorted(turns, key=lambda x: x[0])]

# Display one example
if len(msgs) >= 4:
    i = random.randint(0, len(msgs) - 4)
    input_text  = "\n\n".join(msgs[i:i+3])
    target_text = msgs[i+3]

    print("▶ Thread title:", title)
    print("\nInput (previous 3 messages):\n")
    print(input_text)
    print("\nTarget (next message):\n")
    print(target_text)
else:
    print("Thread too short:", title)


In [ ]:
# Sample a row from train_pairs
i = randint(0, len(train_pairs) - 1)
sample = train_pairs[i]

print("Input text:")
print(sample["input_text"])
print("\nTarget text:")
print(sample["target_text"])

# ------------------------------------------------------------------
# 3 · Save to disk

In [ ]:
for folder in ["bluemoon_train_ds", "bluemoon_test_ds"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

train_pairs.save_to_disk("bluemoon_train_ds")
test_pairs.save_to_disk("bluemoon_test_ds")

print("✅ wrote", len(train_pairs), "train examples and",
      len(test_pairs), "test examples.")

# ------------------------------------------------------------------
# 4 · Tokenization

In [ ]:
# ─── Tokenization Cell ─────────────────────────────────────────────────────
from datasets import load_from_disk
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import os

# 1) Load GPT-2 tokenizer and ensure it has a pad token
tok = AutoTokenizer.from_pretrained("gpt2", trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# 2) Define encoding function
def encode(batch):
    # Tokenize context and target separately
    enc = tok(batch["input_text"],
              truncation=True,
              padding="longest",
              max_length=2048)
    lab = tok(batch["target_text"],
              truncation=True,
              padding="longest",
              max_length=512)
    return {
        "input_ids":      enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels":         lab["input_ids"],
    }

# 3) Process and save both splits
for split in ["train", "test"]:
    ds = load_from_disk(f"bluemoon_{split}_ds")
    ds_tok = ds.map(
        encode,
        batched=True,
        remove_columns=ds.column_names,
        num_proc=8,
        desc=f"Tokenizing {split}"
    )
    out_dir = f"bluemoon_{split}_tok_ds"
    if os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)
    ds_tok.save_to_disk(out_dir)


# ------------------------------------------------------------------
# 5 · train mini deepseek

In [ ]:
import importlib, mini_deepseek
importlib.reload(mini_deepseek)
from mini_deepseek import RolePlayTransformer

In [ ]:
DATA_DIR = "bluemoon_train_tok_ds"   # ← change if needed
CKPT_DIR = "mini_ds_ckpts"           # checkpoints go here
#tokenizer_name = "deepseek-ai/DeepSeek-V3"
tokenizer_name = "gpt2"

# 2 ▸ import your model class
from mini_deepseek import RolePlayTransformer
from transformers import AutoTokenizer
import torch, os, math

tok   = AutoTokenizer.from_pretrained("gpt2", trust_remote_code=True)
tok.pad_token = tok.eos_token
model = RolePlayTransformer(
    vocab=len(tok),
    gradient_checkpointing=True,
)          # keep defaults or tweak dims

os.makedirs(CKPT_DIR, exist_ok=True)                 # make sure folder exists

# 5 ▸ train (prints loss every 100 steps, saves each epoch)
model.train_model(
    dataset_path = DATA_DIR,         # ← tokenised Arrow dataset folder
    tokenizer_name = "gpt2",
    epochs = 5,
    micro_batch = 4,
    grad_accum   = 8,   # effective 32
    lr = 1.5e-4,
    warmup_updates = 1000,
    device = "cuda" if torch.cuda.is_available() else "cpu",
    save_path = CKPT_DIR,
)


# ------------------------------------------------------------------
# 6 · Perplexity of mini deepseek


In [ ]:
import torch

from mini_deepseek import RolePlayTransformer
#from markov_baseline import MarkovChain


In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
import gc

# 1. Delete large objects if they exist
for var in ['model', 'optimizer', 'loader', 'ds']:
    if var in globals():
        try:
            del globals()[var]
        except:
            pass

# 2. Run garbage collector
gc.collect()

# 3. Clear PyTorch CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
DATA_DIR = "bluemoon_test_tok_ds"
device   = "cuda" if torch.cuda.is_available() else "cpu"

model = RolePlayTransformer(vocab=len(tok), gradient_checkpointing=True).to(device)
model.load_state_dict(torch.load(f"ckpt_ep2.pt", map_location=device, weights_only=False))
model.eval()

In [ ]:
ppl_transformer = model.evaluate_perplexity(
    dataset_path=DATA_DIR,
    tokenizer=tok,
    device=device,
    batch_size=8
)
print(f"Transformer perplexity: {ppl_transformer:.2f}")


In [ ]:
# 1) Simple greeting exchange
prev_msgs = [
    "Hi there!",
    "Hello! How can I help you today?",
    "Can you tell me a joke?"
]

reply = model.generate_chat(prev_msgs, max_new_tokens=30, temperature=0.8, top_k = 50)
print(reply)


In [ ]:
# 2) Trivia question
prev_msgs = [
    "What's the tallest mountain in the world?",
    "Mount Everest is the tallest mountain above sea level.",
    "And which mountain is the tallest overall, including base below sea level?"
]
reply = model.generate_chat(prev_msgs, max_new_tokens=50, temperature=0.7)
print(reply)


In [ ]:
# 3) Creative story prompt
prev_msgs = [
    "User: Once upon a time, a curious cat found a mysterious key.",
    "Assistant: The cat examined the key closely—it glowed with a faint blue light.",
    "User: What happened next?"
]
reply = model.generate_chat(prev_msgs, max_new_tokens=60, temperature=1.0)
print(reply)


In [ ]:
batch_prompts = [
    ["User: Hello", "Assistant: Hi! What's up?", "User: Tell me a fun fact."],
    ["User: Define recursion.", "Assistant: Recursion is ...", "User: Can you give me a simple example?"],
    ["User: Who won the World Cup in 2018?", "Assistant: France did.", "User: And in 2022?"]
]

for prev in batch_prompts:
    print("\n")
    print(">>> Prompt:", prev[-1])
    print(model.generate_chat(prev, max_new_tokens=30, temperature=0.8))
    print()


In [ ]:
mc = MarkovChain(n=2)
mc.train_dataset(DATA_DIR, text_field="input_text")

ppl_markov = mc.perplexity_dataset(DATA_DIR, text_field="input_text")
print(f"Markov baseline perplexity: {ppl_markov:.2f}")

In [ ]:
prev_msgs = [
    "User: Hi there!",
    "Assistant: Hello! How can I help you today?",
    "User: Can you tell me a joke?"
]
reply = mc.generate_chat(prev_msgs)
print(reply)

In [ ]:
prev_msgs = [
    "User: Once upon a time, a curious cat found a mysterious key.",
    "Assistant: The cat examined the key closely—it glowed with a faint blue light.",
    "User: What happened next?"
]
reply = mc.generate_chat(prev_msgs)
print(reply)

In [ ]:
prev_msgs = [
    "User: What's the tallest mountain in the world?",
    "Assistant: Mount Everest is the tallest mountain above sea level.",
    "User: And which mountain is the tallest overall, including base below sea level?"
]
reply = mc.generate_chat(prev_msgs)
print(reply)

In [ ]:
batch_prompts = [
    ["User: Hello", "Assistant: Hi! What's up?", "User: Tell me a fun fact."],
    ["User: Define recursion.", "Assistant: Recursion is ...", "User: Can you give me a simple example?"],
    ["User: Who won the World Cup in 2018?", "Assistant: France did.", "User: And in 2022?"]
]

for prev in batch_prompts:
    print(">>> Prompt:", prev[-1])
    print(mc.generate_chat(prev_msgs))
    print()
